# ElectraGuard starter

## Objective
Explore the fictional polling-unit reference, flag individual result sheets for review, and evaluate a text triage baseline on held-out synthetic reports.

Install the packages in `backend/requirements.txt` for backend reuse. From the repository root run `jupyter nbconvert --to notebook --execute notebooks/electraguard_starter.ipynb --output electraguard_executed.ipynb`. The notebook works without a database. See the [dataset licence and provenance](../data/synthetic/README.md).

## Load the synthetic starter data

In [ ]:
import csv
import json
import os
from collections import Counter
from pathlib import Path
import re
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "backend" / "manage.py").is_file())
data = root / "data" / "synthetic"

def csv_rows(path):
    with path.open(encoding="utf-8-sig", newline="") as source:
        return list(csv.DictReader(source))

reference = json.loads((root / "data/reference/official-synthetic-areas.json").read_text(encoding="utf-8"))
units = {
    code: (state["name"], lga["name"], ward["name"], name)
    for state in reference["states"]
    for lga in state["lgas"]
    for ward in lga["wards"]
    for code, name in ward["units"]
}
official_units = csv_rows(data / "polling_units_synthetic.csv")
assert len(units) == len(official_units) == 900
print(f'Reference {reference["dataset_version"]}: {len(units)} synthetic polling units')
for code, location in list(units.items())[:5]:
    print(code, *location, sep=" | ")

## Result-sheet consistency

Flags prompt review of one fictional sheet at a time. The evaluation treats any non-`none` held-out label as a positive and displays only flag counts and detection metrics. No cross-unit vote totals are produced.

In [ ]:
sys.path.insert(0, str(root / "backend"))
try:
    from reports.result_checks import check_result_sheet
    sheet_method = "backend consistency checks"
except ModuleNotFoundError as missing:
    if missing.name != "reports.result_checks":
        raise

    def check_result_sheet(sheet):
        flags = []
        if sheet["accredited"] > sheet["registered"]:
            flags.append({"code": "accredited_exceeds_registered"})
        if sum(sheet["votes"].values()) + sheet["rejected"] != sheet["accredited"]:
            flags.append({"code": "count_mismatch"})
        if sum(v >= 10 and v % 10 == 0 for v in sheet["votes"].values()) >= 3:
            flags.append({"code": "round_numbers"})
        return flags

    sheet_method = "standalone baseline (accreditation, ballot count, rounded entries)"

def ratio(n, d):
    return f"{n / d:.1%} ({n}/{d})" if d else "undefined (0/0)"

sheets = csv_rows(data / "results_synthetic.csv")
labels = {r["pu_id"]: r["anomaly_type"] for r in csv_rows(data / "anomaly_labels_HELD_OUT.csv")}
assert {row["pu_id"] for row in sheets} == units.keys() == labels.keys()
flag_counts, outcomes = Counter(), Counter()
for row in sheets:
    sheet = {
        "registered": int(row["registered_voters"]),
        "accredited": int(row["accredited_voters"]),
        "rejected": int(row["rejected_votes"]),
        "votes": {key: int(value) for key, value in row.items() if key.startswith("votes_")},
    }
    flags = check_result_sheet(sheet)
    flag_counts.update(flag["code"] for flag in flags)
    outcomes[labels[row["pu_id"]] != "none", bool(flags)] += 1
tp, fp, fn = outcomes[True, True], outcomes[False, True], outcomes[True, False]
print(sheet_method, dict(flag_counts))
print("Detection precision:", ratio(tp, tp + fp), "| recall:", ratio(tp, tp + fn))

## Triage category evaluation

The backend baseline uses its pre-existing lexical examples. The standalone fallback learns word sets from the development split. The triage corpus is separate from the official result-sheet corpus, with incident IDs checked for split leakage. These metrics describe synthetic examples.

In [ ]:
triage_rows = csv_rows(root / "evaluation/synthetic_triage.csv")
development = [row for row in triage_rows if row["split"] == "development"]
heldout = [row for row in triage_rows if row["split"] == "heldout"]
assert {r["incident"] for r in development}.isdisjoint(r["incident"] for r in heldout)
try:
    import django
except ModuleNotFoundError as missing:
    if missing.name != "django":
        raise
    django = None

if django is not None:
    os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
    os.environ.setdefault("DJANGO_SECRET_KEY", "local-notebook-unused")
    os.environ.setdefault("DJANGO_DEBUG", "true")
    for key in ("POSTGRES_DB", "POSTGRES_USER", "POSTGRES_PASSWORD"):
        os.environ.setdefault(key, "unused")
    django.setup()
    from reports.triage import METHOD_VERSION, categorize
else:
    METHOD_VERSION = "standalone development-only word-overlap baseline"

    def words(text):
        return set(re.findall(r"\w{4,}", text.casefold()))

    training = [(row["category"], words(row["description"])) for row in development]

    def categorize(text):
        sample = words(text)
        score, label = max(
            (len(sample & known) / max(1, len(sample | known)), category)
            for category, known in training
        )
        return label if score else "other"

pairs = [(row["category"], categorize(row["description"])) for row in heldout]
print(METHOD_VERSION, "|", len(heldout), "held-out reports")
print("Accuracy:", ratio(sum(actual == predicted for actual, predicted in pairs), len(pairs)))
for label in sorted({actual for actual, _ in pairs}):
    correct = pairs.count((label, label))
    print(label, "precision", ratio(correct, sum(p == label for _, p in pairs)),
          "| recall", ratio(correct, sum(a == label for a, _ in pairs)))